# 6.7 一個 token 一定是一個完整字嗎？


「貓🙂」第一byte單獨解碼顯示�，完整七byte卻還原原文。差別在取了多少材料，不一定是原資料壞了。

貓的UTF-8是232、178、147，🙂是240、159、153、130。BPE按頻率合併，邊界未必對齊完整字；token可以是字的一部分，也可以含多字。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
raw = "貓🙂".encode()
print("全部bytes", list(raw))
print("只看第一byte", raw[:1].decode("utf-8", errors="replace"))
print("先拼完整再解碼", raw.decode("utf-8"))
assert raw.decode("utf-8") == "貓🙂"

全部bytes [232, 178, 147, 240, 159, 153, 130]
只看第一byte �
先拼完整再解碼 貓🙂


輸入完整原句並轉成bytes；Python字串的`.encode()`預設使用UTF-8，所以這裡不必另填編碼名稱。第一行應 `[232,178,147,240,159,153,130]`。前三個一起表示貓，後四個一起表示🙂。`raw[:1]` 只留232，UTF-8解碼器知道它是多byte字的開頭，卻沒有後續材料；`errors="replace"` 遇不完整資料就用替代字符�顯示，因此第二行是�。沒有replace時，這段不完整材料會報Unicode解碼錯誤。

第三行解完整七byte，還原貓🙂，assert通過。這說明第一次看到�是觀察方式造成的，不足以證明原始bytes已經被改壞。相反，若把這個�當成真正內容再存回，就已經用替代字符丟失了原來那部分資訊，之後拼接也不會恢復。應保留原bytes或完整token序列，不把臨時顯示的替代符號當原始資料。

生成可能先給半字，下一步補齊。介面可延後解碼或保留未完成尾段，不能每token獨立decode再接替代符號。保留原始bytes和完整token列，才能把暫時觀察不完整與資料真正丟失區分開。

練習只把第二行的切片改成 `raw[:3]`，先預測剛好涵蓋完整貓，顯示貓；再試 `raw[:4]`，預測貓後接�，因為emoji只取到開頭一byte。最後完整decode仍應貓🙂，三次觀察用同一份原資料，便能定位問題在邊界而非內容。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
